# Brazil city finance context: build and validation

This notebook builds the release's record and match tables from the reviewed source datasets. It reads each source review's clean table and its draft action mapping, keeps links to the 32 in-scope adaptation actions, maps each source into one shared project schema, validates the result and writes the two tables to `data/`. Cost band remains the only finance field used to rank actions; everything built here is context. See `review.md` for what the tables can and cannot support.

In [1]:
import csv
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
repo = next((p for p in [start, *start.parents] if (p / 'dataset-review').is_dir()), None)
assert repo is not None, 'Run inside the CityCatalyst-global-data repository'
REVIEWS = repo / 'dataset-review/reviews'
RELEASE = REVIEWS / 'oef/br-city-finance-feasibility/releases/v2'
DATA = RELEASE / 'data'      # committed: records whose source licence allows redistribution
SAMPLE = RELEASE / 'sample'  # gitignored: full tables, including sources not cleared for redistribution

# Sources whose record-level reuse terms are unresolved (see reuse_conditions in datasets.csv).
# Their files live in each source review's gitignored sample/ folder; they are used for analysis but never committed.
RESTRICTED = {'Novo PAC drainage and slope-containment selections', 'COFIEX external-financing meeting results',
              'CGU civil-defence transfer payments'}
skipped = []

ILLUSTRATIVE = 'Illustrative only - not a final implementation recommendation'
CIVIL_THEMES = {'disaster_risk', 'slope_stabilisation'}  # Transferegov screens read as civil-defence finance

def read(rel):
    """Read a source review file as text, so identifiers and amounts pass through unchanged."""
    return pd.read_csv(REVIEWS / rel, dtype=str, keep_default_na=False, low_memory=False)

print('release folder:', RELEASE.relative_to(repo))

release folder: dataset-review/reviews/oef/br-city-finance-feasibility/releases/v2


## 1. Actions in scope

The action input is the V13 final adaptation action bank (37 actions). An action stays in scope only when the adaptation impact review holds at least one component-level claim for it, which leaves 32. Links to any other action are dropped in every source below.

In [2]:
bank = pd.read_csv(DATA / 'adaptation_action_bank_v13.csv', header=1, dtype=str, keep_default_na=False)
bank = bank[bank['ID'].str.strip() != '']
impact = json.loads((REVIEWS / 'oef/br-action-adaptation-impact/releases/v1/output/action_risk_assessments.json').read_text())
scored = {a['action_id'] for a in impact['actions'] if a['component_claims']}
actions = bank[bank['ID'].isin(scored)].set_index('ID')['Action']
assert len(bank) == 37 and len(actions) == 32
print(f'{len(bank)} actions in V13; {len(actions)} in scope; left out: {sorted(set(bank["ID"]) - set(actions.index))}')

37 actions in V13; 32 in scope; left out: ['c40_0046', 'icare_0165', 'icare_0185', 'new0001', 'new0002']


## 2. Shared rules

Every source is mapped into the same project columns. Location reads "place, state" and falls back to the state alone; codes outside the state list pass through unchanged. Match labels, strength and review status are relabelled from each mapping file, and mapping rows with no confidence are dropped. `approved_amount` is the funder's own amount at the stage the record describes (contracted, committed, requested, selected or paid). `disbursed_amount` is filled only where the source reports money actually paid out, which is BNDES (disbursed) and FNMC (transferred); it is left blank everywhere else rather than filled with a planned or total-cost figure.

In [3]:
UF = {'AC': 'Acre', 'AL': 'Alagoas', 'AM': 'Amazonas', 'AP': 'Amapá', 'BA': 'Bahia', 'CE': 'Ceará', 'DF': 'Federal District',
      'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão', 'MG': 'Minas Gerais', 'MS': 'Mato Grosso do Sul', 'MT': 'Mato Grosso',
      'PA': 'Pará', 'PB': 'Paraíba', 'PE': 'Pernambuco', 'PI': 'Piauí', 'PR': 'Paraná', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
      'RO': 'Rondônia', 'RR': 'Roraima', 'RS': 'Rio Grande do Sul', 'SC': 'Santa Catarina', 'SE': 'Sergipe', 'SP': 'São Paulo',
      'TO': 'Tocantins', 'BR': 'Brazil'}
ROUTE = {'grant': 'Domestic climate and environmental grants', 'credit': 'Domestic climate and infrastructure credit',
         'federal': 'Federal transfers and public investment programmes', 'external': 'Direct external development-bank borrowing',
         'facility': 'International grant facilities', 'civil': 'Civil-defence and disaster finance'}
MATCH_TYPE = {'direct': 'Direct match', 'component': 'Component match', 'related': 'Related match', 'enabling': 'Enabling-condition match'}
STRENGTH = {'high': 'High', 'medium': 'Medium'}
REVIEW = {'High': 'Draft match - no additional mapping review flagged', 'Medium': 'Human review pending'}

def location(place, uf):
    state = UF.get(uf, uf)
    place = place.strip()
    if place and place.casefold() != state.casefold():
        return f'{place}, {state}'
    return place or state

def locations(places, ufs):
    return [location(a, b) for a, b in zip(places, ufs)]

def first(preferred, fallback):
    return preferred.where(preferred != '', fallback)

def links(rel, rename=None):
    """Read a mapping file into key, action_id, match_type, confidence, rationale."""
    return read(rel).rename(columns={'mapping_confidence': 'confidence', **(rename or {})})

projects, matches = {}, {}

def add(dataset, records, mapping, key, reference):
    """Keep records with at least one in-scope link, and build their match rows."""
    mapping = mapping[(mapping['confidence'] != 'none') & mapping['action_id'].isin(actions.index)]
    kept = records[records[key].isin(mapping[key])].copy()
    kept['project_reference'] = reference(kept)
    kept['dataset'] = dataset
    joined = kept.merge(mapping[[key, 'action_id', 'match_type', 'confidence', 'rationale']], on=key)
    projects[dataset] = kept
    matches[dataset] = pd.DataFrame({
        'project_reference': joined['project_reference'], 'project_name': joined['project_name'],
        'action_id': joined['action_id'], 'action_name': joined['action_id'].map(actions),
        'match_type': joined['match_type'].map(MATCH_TYPE).fillna(joined['match_type']),
        'match_strength': joined['confidence'].map(STRENGTH), 'why_it_matches': joined['rationale']})
    return pd.Series({'source rows': len(records), 'records kept': len(kept), 'matches': len(joined)}, name=dataset).to_frame().T

## 3. Sources

One block per source. Each maps the source review's clean table into the project columns and joins its draft action mapping. BNDES, SADIPEM and CGU are mapped by description, purpose and budget action rather than by record, so one mapping row can reach many records.

In [4]:
# BNDES non-automatic operations: mapping keyed on the Portuguese project description
s = read('br-bndes/br-bndes-operacoes-financiamento/releases/2026-09-01/data/br_bndes_operacoes_financiamento_subcredits.csv.gz')
records = pd.DataFrame({
    'key': s.project_description_pt, 'id': s.raw_row_id, 'project_name': s.project_description, 'project_name_pt': s.project_description_pt,
    'record_type': 'Financing operation', 'recipient': s.client_name, 'location': locations(s.municipality_name, s.state_code),
    'finance_route': s.support_modality.map({'Non-repayable': ROUTE['grant'], 'Repayable': ROUTE['credit']}),
    'finance_instrument': s.financing_instrument, 'approved_amount': s.contracted_amount_brl, 'currency': 'BRL',
    'disbursed_amount': s.disbursed_amount_brl, 'status': s.contract_status.replace('', 'Not stated'), 'decision_date': s.contract_date,
    'source': 'https://dadosabertos.bndes.gov.br/dataset/operacoes-financiamento'})
assert records.finance_route.notna().all()
mapping = links('br-bndes/br-bndes-operacoes-financiamento/releases/2026-09-01/data/br_bndes_operacoes_financiamento_to_adaptation_actions.csv', {'src_description_pt': 'key'})
add('BNDES non-automatic financing operations', records, mapping, 'key', lambda d: 'BNDES ' + d.id)

,source rows,records kept,matches
BNDES non-automatic financing operations,23815,854,920


In [5]:
# FNMA transfer instruments: record-level mapping
# disbursed_amount stays blank: the source holds planned amounts only
s = read('br-mma/br-fnma-instrumentos/releases/2025-03/data/br_fnma_transfer_instruments.csv')
records = pd.DataFrame({
    'record_id': s.record_id, 'project_name': first(s.project_title, s.project_title_pt), 'project_name_pt': s.project_title_pt,
    'record_type': 'Transfer agreement', 'recipient': s.executing_institution_name,
    'location': locations(s.executing_institution_city_name, s.state_code), 'finance_route': ROUTE['grant'],
    'finance_instrument': first(s.transfer_instrument_id, s.internal_instrument_id), 'approved_amount': s.fnma_planned_transfer_amount_brl,
    'currency': 'BRL', 'disbursed_amount': '', 'status': 'Not stated', 'decision_date': s.signature_date,
    'source': 'https://dados.mma.gov.br/dataset/fundo-nacional-do-meio-ambiente'})
mapping = links('br-mma/br-fnma-instrumentos/releases/2025-03/data/br_fnma_transfer_instruments_to_adaptation_actions.csv')
add('FNMA transfer instruments', records, mapping, 'record_id', lambda d: 'FNMA ' + d.record_id)

,source rows,records kept,matches
FNMA transfer instruments,1495,465,465


In [6]:
# FNMC non-reimbursable projects: record-level mapping; state_code can hold several states
# approved = the fund's committed share; disbursed = the amount transferred
s = read('br-mma/br-fnmc-projetos/releases/2025-11/data/br_fnmc_projects.csv')
records = pd.DataFrame({
    'record_id': s.record_id, 'project_name': s.project_name, 'project_name_pt': s.project_name_pt, 'record_type': 'Project',
    'recipient': s.executing_institution, 'location': locations([''] * len(s), s.state_code), 'finance_route': ROUTE['grant'],
    'finance_instrument': s.transfer_instrument_id, 'approved_amount': s.fnmc_committed_amount_brl, 'currency': 'BRL',
    'disbursed_amount': s.transferred_amount_brl, 'status': 'Not stated', 'decision_date': s.agreement_start_date, 'source': s.source_url})
mapping = links('br-mma/br-fnmc-projetos/releases/2025-11/data/br_fnmc_projects_to_adaptation_actions.csv')
add('FNMC non-reimbursable projects', records, mapping, 'record_id', lambda d: 'FNMC ' + d.record_id)

,source rows,records kept,matches
FNMC non-reimbursable projects,82,23,23


In [7]:
# Novo PAC drainage and slope selections: record-level mapping, finance-compile schema
folder = 'br-mcidades/br-novopac-drenagem-encostas/releases/2025-09/sample/'
if (REVIEWS / folder).is_dir():
    s = read(folder + 'br_novopac_drainage_slopes.csv')
    records = pd.DataFrame({
        'record_id': s.record_id, 'project_name': s.project_description, 'project_name_pt': s.project_description_pt,
        'record_type': 'Programme selection', 'recipient': s.recipient, 'location': locations(s.municipality_name, s.state_code),
        'finance_route': ROUTE['federal'],
        'finance_instrument': s.instrument.map({'grant': 'Grant', 'loan-domestic': 'Domestic loan', 'grant+loan': 'Grant and domestic loan'}),
        'approved_amount': s.amount, 'currency': s.currency, 'disbursed_amount': '',
        'status': s.status.map({'selected': 'Selected', 'selected-aggregate': 'Selected (aggregate)'}), 'decision_date': s.decision_date,
        'source': s.source_url})
    mapping = links(folder + 'br_novopac_drainage_slopes_to_adaptation_actions.csv')
    add('Novo PAC drainage and slope-containment selections', records, mapping, 'record_id', lambda d: 'Novo PAC ' + d.record_id)
else:
    skipped.append('Novo PAC drainage and slope-containment selections')
    print('Skipped: held only in the source review sample/ folder')

In [8]:
# COFIEX meeting results: record-level mapping, finance-compile schema; a decision authorises preparation, not a loan
folder = 'br-mpo/br-cofiex-meeting-results/releases/2025-2026/sample/'
if (REVIEWS / folder).is_dir():
    s = read(folder + 'br_cofiex_meeting_results.csv')
    records = pd.DataFrame({
        'record_id': s.record_id, 'project_name': s.project_description, 'project_name_pt': s.project_description_pt,
        'record_type': 'External-finance preparation decision', 'recipient': s.recipient, 'location': locations(s.municipality_name, s.state_code),
        'finance_route': ROUTE['external'], 'finance_instrument': s.instrument, 'approved_amount': s.amount, 'currency': s.currency,
        'disbursed_amount': '', 'status': s.status.map({'cofiex-approved': 'Approved for preparation', 'cofiex-not-approved': 'Not approved'}),
        'decision_date': s.decision_date, 'source': s.source_url})
    mapping = links(folder + 'br_cofiex_meeting_results_to_adaptation_actions.csv')
    add('COFIEX external-financing meeting results', records, mapping, 'record_id', lambda d: 'COFIEX ' + d.record_id)
else:
    skipped.append('COFIEX external-financing meeting results')
    print('Skipped: held only in the source review sample/ folder')

In [9]:
# SADIPEM credit requests: mapping keyed on the Portuguese purpose; external operations go to the external-borrowing route
s = read('br-stn/br-sadipem-operacoes-credito/releases/2026-09-20/data/br_sadipem_credit_operations.csv.gz')
records = pd.DataFrame({
    'key': s.purpose_pt, 'id': s.pvl_number, 'project_name': first(s.project_description, s.project_description_pt),
    'project_name_pt': s.project_description_pt, 'record_type': 'Credit request', 'recipient': s.interested_party,
    'location': locations(s.interested_party, s.state_code),
    'finance_route': [ROUTE['external'] if t.startswith('External') else ROUTE['credit'] for t in s.operation_type],
    'finance_instrument': s.operation_type, 'approved_amount': s.amount, 'currency': s.currency, 'disbursed_amount': '',
    'status': s.status, 'decision_date': s.status_date, 'source': s.source_url})
mapping = links('br-stn/br-sadipem-operacoes-credito/releases/2026-09-20/data/br_sadipem_purposes_to_adaptation_actions.csv', {'purpose_pt': 'key'})
add('SADIPEM credit-operation requests', records, mapping, 'key', lambda d: 'SADIPEM ' + d.id)

,source rows,records kept,matches
SADIPEM credit-operation requests,28402,1231,1231


In [10]:
# CGU civil-defence payments: mapping keyed on budget action code; the reference is the 1-based source row
payments = 'br-cgu/br-transparencia-transferencias/releases/2026-07/sample/br_transparencia_transferencias_civil_defence_payments.csv'
if (REVIEWS / payments).is_file():
    s = read(payments)
    records = pd.DataFrame({
        'key': s.action_code, 'id': [str(i + 1) for i in range(len(s))], 'project_name': s.action_name, 'project_name_pt': s.action_name_pt,
        'record_type': 'Transfer payment', 'recipient': s.beneficiary_name, 'location': locations(s.municipality_name, s.uf),
        'finance_route': ROUTE['civil'], 'finance_instrument': s.transfer_type, 'approved_amount': s.amount_transferred_brl, 'currency': 'BRL',
        'disbursed_amount': '', 'status': 'Not stated', 'decision_date': s.transfer_year_month,
        'source': 'https://portaldatransparencia.gov.br/download-de-dados/transferencias'})
    mapping = links('br-cgu/br-transparencia-transferencias/releases/2026-07/data/br_transparencia_transferencias_to_adaptation_actions.csv', {'src_action_code': 'key'})
    add('CGU civil-defence transfer payments', records, mapping, 'key', lambda d: 'CGU row-' + d.id)
else:
    skipped.append('CGU civil-defence transfer payments')
    print('Skipped: held only in the source review sample/ folder')

In [11]:
# Transferegov: five theme-level screens with no records or amounts behind them yet
mapping = links('br-mgi/br-transferegov-discricionarias-legais/releases/2026-07-17/data/br_transferegov_theme_to_adaptation_actions.csv').assign(match_type='Theme-level screen')
records = pd.DataFrame({
    'screen_label': mapping.screen_label, 'project_name': mapping.screen_label.str.replace('_', ' ').str.title(), 'project_name_pt': '',
    'record_type': 'Theme-level match', 'recipient': '', 'location': 'Brazil',
    'finance_route': [ROUTE['civil'] if t in CIVIL_THEMES else ROUTE['federal'] for t in mapping.screen_label],
    'finance_instrument': 'Theme-level match', 'approved_amount': '', 'currency': '', 'disbursed_amount': '',
    'status': 'Proposal matching pending', 'decision_date': '', 'source': 'https://www.gov.br/transferegov/pt-br'})
add('Transferegov discretionary and legal transfers', records, mapping, 'screen_label', lambda d: 'Transferegov ' + d.screen_label)

,source rows,records kept,matches
Transferegov discretionary and legal transfers,5,5,5


In [12]:
# IDB Brazil adaptation projects: technical cooperation goes to the grant-facility route
# disbursed_amount stays blank: the source has total project cost, not disbursement
s = read('idb/idb-projects/releases/2026-08-25/data/idb_brazil_adaptation_projects.csv')
instrument = first(s.lending_instrument_name, s.operation_type_name)
records = pd.DataFrame({
    'project_id': s.project_id, 'project_name': s.project_name, 'project_name_pt': '', 'record_type': 'Project',
    'recipient': s.country_name, 'location': 'Brazil',
    'finance_route': [ROUTE['facility'] if i == 'Technical Cooperation' else ROUTE['external'] for i in instrument],
    'finance_instrument': instrument, 'approved_amount': s.original_approved_amount_usd, 'currency': 'USD',
    'disbursed_amount': '', 'status': s.public_status, 'decision_date': s.approval_date,
    'source': 'https://data.iadb.org/dataset/idb-projects-dataset'})
mapping = links('idb/idb-projects/releases/2026-08-25/data/idb_brazil_adaptation_projects_to_actions.csv')
add('IDB Brazil adaptation projects', records, mapping, 'project_id', lambda d: 'IDB ' + d.project_id)

,source rows,records kept,matches
IDB Brazil adaptation projects,63,36,47


In [13]:
# World Bank Brazil adaptation candidates: recipient-executed activities go to the grant-facility route
# disbursed_amount stays blank: the source has project cost, not disbursement
s = read('world-bank/world-bank-projects/releases/v3/data/world_bank_brazil_adaptation_projects.csv')
records = pd.DataFrame({
    'project_id': s.project_id, 'project_name': s.project_name, 'project_name_pt': s.project_name_pt, 'record_type': 'Project',
    'recipient': s.borrower_name, 'location': 'Brazil',
    'finance_route': [ROUTE['facility'] if pl == 'Recipient Executed Activities' else ROUTE['external'] for pl in s.product_line_name],
    'finance_instrument': s.lending_instrument, 'approved_amount': s.total_commitment_amount_usd, 'currency': 'USD',
    'disbursed_amount': '', 'status': s.lifecycle_status, 'decision_date': s.board_approval_date, 'source': s.project_url})
mapping = links('world-bank/world-bank-projects/releases/v3/data/world_bank_brazil_adaptation_projects_to_actions.csv',
                {'relationship': 'match_type', 'mapping_rationale': 'rationale'})
add('World Bank Brazil adaptation candidate projects', records, mapping, 'project_id', lambda d: 'World Bank ' + d.project_id)

,source rows,records kept,matches
World Bank Brazil adaptation candidate projects,55,9,9


## 4. Assemble and validate

The ten sources are stacked into the two release tables and sorted by reference. The checks run before anything is written: if one fails, nothing is exported.

In [14]:
PROJECT_COLUMNS = ['project_reference', 'project_name', 'project_name_pt', 'record_type', 'recipient', 'location', 'finance_route',
                   'finance_instrument', 'approved_amount', 'currency', 'disbursed_amount', 'status', 'decision_date', 'dataset', 'source']
MATCH_COLUMNS = ['project_reference', 'project_name', 'action_id', 'action_name', 'match_type', 'match_strength', 'match_status',
                 'review_status', 'why_it_matches']

project_table = (pd.concat(projects.values())[PROJECT_COLUMNS]
                 .sort_values('project_reference', kind='stable').reset_index(drop=True))
match_table = pd.concat(matches.values()).sort_values(['project_reference', 'action_id'], kind='stable').reset_index(drop=True)
match_table['match_status'] = ILLUSTRATIVE
match_table['review_status'] = match_table['match_strength'].map(REVIEW)
match_table = match_table[MATCH_COLUMNS]

datasets = pd.read_csv(DATA / 'datasets.csv', dtype=str, keep_default_na=False)
routes = pd.read_csv(DATA / 'routes.csv', dtype=str, keep_default_na=False)

pd.DataFrame({'records': project_table.dataset.value_counts(),
              'matches': match_table.merge(project_table[['project_reference', 'dataset']]).dataset.value_counts()})

,records,matches
dataset,,
SADIPEM credit-operation requests,1231,1231
BNDES non-automatic financing operations,854,920
CGU civil-defence transfer payments,743,746
FNMA transfer instruments,465,465
Novo PAC drainage and slope-containment selections,317,317
IDB Brazil adaptation projects,36,47
FNMC non-reimbursable projects,23,23
COFIEX external-financing meeting results,17,17
World Bank Brazil adaptation candidate projects,9,9


In [15]:
# Structure
assert project_table.project_reference.is_unique
assert (project_table != '').all().loc[['project_reference', 'project_name', 'record_type', 'finance_route', 'status', 'dataset', 'source']].all()
assert project_table.notna().all().all() and match_table.notna().all().all()
for field in ['approved_amount', 'disbursed_amount']:
    pd.to_numeric(project_table.loc[project_table[field] != '', field])  # raises if any amount is not numeric
assert set(project_table.currency) <= {'BRL', 'USD', 'EUR', 'JPY', ''}
assert project_table.source.str.startswith('http').all()

# Joins between the four tables
assert set(match_table.project_reference) == set(project_table.project_reference), 'every record needs at least one match'
assert set(project_table.finance_route) == set(routes.finance_route)
complete = not skipped
if complete:
    # records_used in datasets.csv counts the full catalogue, so it can only be checked when every source is present
    assert set(project_table.dataset) == set(datasets.dataset)
    records_used = datasets.set_index('dataset').records_used.astype(int)
    assert records_used.equals(project_table.dataset.value_counts().reindex(records_used.index)), 'datasets.csv records_used is stale'
else:
    assert set(project_table.dataset) == set(datasets.dataset) - set(skipped)
    print('Sources not held locally:', skipped)

# Matching terms
assert set(match_table.action_id) <= set(actions.index)
assert set(match_table.match_type) <= set(MATCH_TYPE.values()) | {'Theme-level screen'}
assert set(match_table.match_strength) == {'High', 'Medium'}
assert (match_table.match_status == ILLUSTRATIVE).all()
assert (match_table.why_it_matches != '').all()

print(f'All checks passed: {len(project_table):,} records, {len(match_table):,} matches, '
      f'{match_table.action_id.nunique()} of {len(actions)} actions matched')

All checks passed: 3,700 records, 3,780 matches, 25 of 32 actions matched


## 5. Export

The committed tables in `data/` hold only records from sources whose licence allows redistribution. When every source is available locally, the full tables, including Novo PAC, COFIEX and CGU records, are also written to the gitignored `sample/` folder for internal analysis.

In [16]:
def write_csv(frame, folder, name):
    folder.mkdir(exist_ok=True)
    with open(folder / name, 'w', encoding='utf-8', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(frame.columns)
        writer.writerows(frame.itertuples(index=False))

shared = ~project_table.dataset.isin(RESTRICTED)
shared_projects = project_table[shared].reset_index(drop=True)
shared_matches = match_table[match_table.project_reference.isin(shared_projects.project_reference)].reset_index(drop=True)
write_csv(shared_projects, DATA, 'projects.csv')
write_csv(shared_matches, DATA, 'project_action_matches.csv')
print(f'data/: {len(shared_projects):,} records and {len(shared_matches):,} matches from sources cleared for redistribution')

if complete:
    write_csv(project_table, SAMPLE, 'projects_all_sources.csv')
    write_csv(match_table, SAMPLE, 'project_action_matches_all_sources.csv')
    print(f'sample/: {len(project_table):,} records and {len(match_table):,} matches from all sources, for internal analysis')

data/: 2,623 records and 2,700 matches from sources cleared for redistribution


sample/: 3,700 records and 3,780 matches from all sources, for internal analysis


## 6. Figures cited in the review

These are the aggregates `review.md` quotes: records by route, the review state of the matches, the currencies in the amount fields, and the benchmark sizes by source. The benchmarks use BRL rows only and the approved-amount field, so each is at the stage its record type describes.

In [17]:
by_route = project_table.groupby(['finance_route', 'record_type']).size().rename('records').reset_index()
display(by_route)
display(match_table.groupby(['match_strength', 'review_status']).size().rename('matches').to_frame())
display(project_table.currency.replace('', '(none)').value_counts().rename('records').to_frame())

,finance_route,record_type,records
0,Civil-defence and disaster finance,Theme-level match,2
1,Civil-defence and disaster finance,Transfer payment,743
2,Direct external development-bank borrowing,Credit request,65
3,Direct external development-bank borrowing,External-finance preparation decision,17
4,Direct external development-bank borrowing,Project,26
5,Domestic climate and environmental grants,Financing operation,149
6,Domestic climate and environmental grants,Project,23
7,Domestic climate and environmental grants,Transfer agreement,465
8,Domestic climate and infrastructure credit,Credit request,1166
9,Domestic climate and infrastructure credit,Financing operation,705


,,matches
match_strength,review_status,
High,Draft match - no additional mapping review flagged,564
Medium,Human review pending,3216


,records
currency,
BRL,3568
USD,116
EUR,7
(none),5
JPY,4


In [18]:
brl = project_table[(project_table.currency == 'BRL') & (project_table.approved_amount != '')].copy()
brl['approved'] = brl.approved_amount.astype(float)
benchmarks = (brl.groupby(['dataset', 'record_type']).approved
              .describe(percentiles=[0.25, 0.5, 0.75])[['count', '25%', '50%', '75%']]
              .query('count >= 10').sort_values('50%').round(-3).astype(int))
benchmarks

,,count,25%,50%,75%
dataset,record_type,,,,
FNMA transfer instruments,Transfer agreement,0,44000,100000,242000
CGU civil-defence transfer payments,Transfer payment,1000,234000,501000,890000
FNMC non-reimbursable projects,Project,0,462000,800000,2206000
SADIPEM credit-operation requests,Credit request,1000,1749000,8001000,25433000
BNDES non-automatic financing operations,Financing operation,1000,3665000,12749000,44677000
Novo PAC drainage and slope-containment selections,Programme selection,0,11043000,23012000,50000000


## Findings

The build reproduces the tables previously assembled outside the repository: the same 3,700 records and 3,780 matches across all ten sources. Every record now traces to a source review file and a stated rule. Because the repository is public, only records from sources cleared for redistribution are committed; Novo PAC, COFIEX and CGU records inform the analysis from the gitignored `sample/` tables.

The amount columns were redefined in this build so that each means one thing across sources. Earlier tables filled `disbursed_amount` with the FNMA total planned amount, the IDB original total cost and the World Bank project cost, and filled the FNMC `approved_amount` with the total project amount including counterpart. `disbursed_amount` is now blank for FNMA, IDB and the World Bank, and FNMC's approved amount is the fund's committed share. No other column changed.

85% of the matches are Medium strength and await human review, and 25 of the 32 in-scope actions have at least one match. The five Transferegov rows are theme-level screens with no underlying records, so the project count includes five rows that are not finance records.

Location text is passed through from each source: 49 BNDES rows carry the code "IE" in place of a state, and multi-state FNMC projects list state codes rather than names.